## INSTALL

In [2]:
import sys

REQUIRED_VERSION = (3, 11, 9)
current_version  = sys.version_info[:3]

if current_version != REQUIRED_VERSION:
    raise RuntimeError(
        f"Wrong Python version. Expected 3.11.9, "
        f"got {'.'.join(str(v) for v in current_version)}. "
        "Make sure the correct kernel is selected."
    )

print(f"Python {'.'.join(str(v) for v in current_version)} OK")

#%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
#%pip install pandas numpy Pillow tqdm scikit-learn xgboost timm
%pip install timm lightgbm catboost

Python 3.11.9 OK
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 9.5 MB/s  0:00:00
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
    --------------------------------------- 2.4/100.2 MB 12.2 MB/s eta 0:00:09
   - -------------------------------------- 4.7/100.2 MB 11.4 MB/s eta 0:00:09
   -- ------------------------------------- 6.8/100.2 MB 11.0 MB/s eta 0:00:09
   --- ------------------------------------ 9.2/100.2 MB 11.2 MB/s eta 0:00:09
   ---- ----------------------------------- 11.5/100.2 MB 11.3 MB/s eta 0:00:08
   ----- ---------------------------------- 13.9/100.2 MB 11.5 MB/s eta 0:00:08
   ------ --------------------------------- 16.5/100.2 MB 11.4 MB/s eta 0:00:08
   ------- -------------------------------- 18.9/100.2 MB 11.5 MB/s eta 0:00:08
   -------- ------------------------------- 21.5/100.2 MB


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## IMPORTS

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    StratifiedKFold, train_test_split,
    cross_val_score, ParameterGrid
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from itertools import product

## MODEL SELECTOR

In [ ]:
import os

# Load HF_TOKEN from the .env file in the project root.
# This avoids hardcoding secrets in the notebook.
with open('.env') as env_file:
    for env_line in env_file:
        if '=' in env_line and not env_line.startswith('#'):
            env_key, env_value = env_line.strip().split('=', 1)
            os.environ[env_key] = env_value

print('HF_TOKEN loaded' if os.environ.get('HF_TOKEN') else 'WARNING: HF_TOKEN not found in .env')

In [ ]:
# --- Model Selector -----------------------------------------------------------
# Pick which backbone extracts image features.
# Changing this requires re-running PREPROCESSING and FEATURE EXTRACTION
# to generate a new set of .pt files for the chosen extractor.
#
#   'dinov2'     — DINOv2 ViT-g/14      (Meta, 1536-d)
#                  weights are pre-trained on LVD-142M
#   'convnextv2' — ConvNeXt V2 Huge     (Meta/FAIR, 2816-d)
#                  weights are pre-trained on ImageNet-22k, fine-tuned on ImageNet-1k at 512px
#
FEATURE_EXTRACTOR = 'dinov2'   # 'dinov2' | 'convnextv2'
# -----------------------------------------------------------------------------

## DEFINITIONS

In [ ]:
# --- Device selection ---------------------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# --- Load backbone based on FEATURE_EXTRACTOR --------------------------------
# Controlled by the MODEL SELECTOR cell above.
# Each extractor saves its features to files with a unique suffix so you can
# switch models without overwriting previously extracted features.

if FEATURE_EXTRACTOR == 'dinov2':
    # DINOv2 ViT-g/14: 1536-d features, trained on LVD-142M
    backbone_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
    backbone_model = backbone_model.to(device).eval()
    backbone_preprocess = transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    def encode(batch):
        return backbone_model(batch.to(device)).float().cpu()

elif FEATURE_EXTRACTOR == 'convnextv2':
    # ConvNeXt V2 Huge: 2816-d features, trained on IN22k then fine-tuned on IN1k at 512px
    import timm
    backbone_model = timm.create_model('convnextv2_huge.fcmae_ft_in22k_in1k_512', pretrained=True, num_classes=0)
    backbone_model = backbone_model.to(device).eval()
    # timm tells us exactly what preprocessing this model expects (crop size, mean, std, etc.)
    convnextv2_data_config = timm.data.resolve_model_data_config(backbone_model)
    backbone_preprocess    = timm.data.create_transform(**convnextv2_data_config, is_training=False)
    def encode(batch):
        return backbone_model(batch.to(device)).float().cpu()

else:
    raise ValueError(f'Unknown FEATURE_EXTRACTOR: {FEATURE_EXTRACTOR!r}. Choose dinov2 or convnextv2.')

# File-name suffix so each extractor saves to its own set of .pt files
FEATURE_SUFFIX_MAP = {'dinov2': '', 'convnextv2': '_convnextv2'}
feat_suffix         = FEATURE_SUFFIX_MAP[FEATURE_EXTRACTOR]
print(f'Backbone: {FEATURE_EXTRACTOR}  |  suffix: {feat_suffix!r}  |  device: {device}')


# --- Image loading and feature extraction helpers ----------------------------

def save_images(image_dir, output_file, has_labels=True):
    # Load every image in a folder, apply backbone_preprocess, and save to a .pt file.
    # has_labels=True  : folder has one subfolder per class
    # has_labels=False : flat folder of images

    image_tensors = []
    labels        = []
    file_paths    = []

    if has_labels:
        class_names    = sorted(os.listdir(image_dir))
        class_to_index = {name: i for i, name in enumerate(class_names)}

        total_images = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )

        count = 0
        for class_name in class_names:
            class_folder = os.path.join(image_dir, class_name)
            for filename in os.listdir(class_folder):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                image_path = os.path.join(class_folder, filename)
                img        = Image.open(image_path).convert('RGB')
                image_tensors.append(backbone_preprocess(img))
                labels.append(class_to_index[class_name])
                file_paths.append(image_path)
                count += 1
                print(f'{count}/{total_images}', end='\r')

        torch.save({
            'tensors':        torch.stack(image_tensors),
            'labels':         torch.tensor(labels),
            'paths':          file_paths,
            'class_to_index': class_to_index
        }, output_file)

    else:
        image_files  = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        total_images = len(image_files)
        for i, filename in enumerate(image_files, start=1):
            image_path = os.path.join(image_dir, filename)
            img        = Image.open(image_path).convert('RGB')
            image_tensors.append(backbone_preprocess(img))
            file_paths.append(image_path)
            print(f'{i}/{total_images}', end='\r')

        torch.save({'tensors': torch.stack(image_tensors), 'paths': file_paths}, output_file)

    print(f'\nSaved {len(image_tensors)} images to {output_file}')


def extract(input_file, output_file):
    # Pass every image through the selected backbone and save feature vectors.
    data         = torch.load(input_file, weights_only=False)
    images       = data['tensors']
    total_images = len(images)
    feature_list = []

    with torch.no_grad():
        for i, image in enumerate(images, start=1):
            # add batch dimension: (3,H,W) -> (1,3,H,W)
            # encode moves the batch to device and returns a CPU tensor
            feature_vector = encode(image.unsqueeze(0)).squeeze().numpy()
            feature_list.append(feature_vector)
            print(f'{i}/{total_images}', end='\r')

    features = np.array(feature_list)
    torch.save({**data, 'features': features}, output_file)
    print(f'\nExtracted {features.shape} -> saved to {output_file}')

In [ ]:
# --- Metrics helper ----------------------------------------------------------

def compute_classification_metrics(true_labels, predicted_labels):
    # Class distribution shows how balanced the split is, which matters when
    # interpreting accuracy — a 90% accuracy means more on a balanced set.
    unique_classes, samples_per_class = np.unique(true_labels, return_counts=True)

    class_distribution_percentages = {
        int(class_index): round(100 * class_sample_count / len(true_labels), 1)
        for class_index, class_sample_count in zip(unique_classes, samples_per_class)
    }

    return {
        'accuracy':      accuracy_score(true_labels, predicted_labels),
        'f1':            f1_score(true_labels, predicted_labels, average='weighted'),
        'precision':     precision_score(true_labels, predicted_labels, average='weighted'),
        'recall':        recall_score(true_labels, predicted_labels, average='weighted'),
        'class_balance': class_distribution_percentages,
    }


def print_cross_validation_report(model_name, fold_metrics_list):
    # Report mean ± std to capture both performance and stability across folds.
    print(f'--- {model_name} ---')

    for metric_name in ('accuracy', 'f1', 'precision', 'recall'):
        metric_values_per_fold = [fold_result[metric_name] for fold_result in fold_metrics_list]
        mean_value = np.mean(metric_values_per_fold)
        std_value  = np.std(metric_values_per_fold)
        print(f'  {metric_name:<9}: {mean_value:.4f} +/- {std_value:.4f}')

    print('  class balance:')
    for class_index, percentage in fold_metrics_list[0]['class_balance'].items():
        print(f'    class {class_index}: {percentage}%')

    print()


# --- MLP model ---------------------------------------------------------------

class MLP(nn.Module):
    # Configurable fully-connected network for classification on top of pre-extracted features.
    def __init__(self, input_size, num_classes, hidden_layers=(256, 128),
                 activation=nn.ReLU, dropout=0.0):
        super().__init__()

        network_layers     = []
        current_layer_size = input_size

        for layer_output_size in hidden_layers:
            network_layers.append(nn.Linear(current_layer_size, layer_output_size))
            network_layers.append(activation())

            if dropout > 0:
                network_layers.append(nn.Dropout(dropout))

            current_layer_size = layer_output_size

        network_layers.append(nn.Linear(current_layer_size, num_classes))
        self.network = nn.Sequential(*network_layers)

    def forward(self, input_tensor):
        return self.network(input_tensor)


# --- Classifier training functions -------------------------------------------
# All functions accept NumPy arrays and return (trained_model, metrics_dict).

def train_logistic_regression(training_features, training_labels,
                               validation_features, validation_labels,
                               regularisation_strength=1.0):
    model = LogisticRegression(C=regularisation_strength, max_iter=2000)
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_support_vector_machine(training_features, training_labels,
                                  validation_features, validation_labels,
                                  kernel='rbf', regularisation_strength=1.0,
                                  gamma='scale'):
    model = SVC(
        kernel=kernel,
        C=regularisation_strength,
        gamma=gamma,
        decision_function_shape='ovr'
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_random_forest(training_features, training_labels,
                         validation_features, validation_labels,
                         number_of_trees=300):
    model = RandomForestClassifier(
        n_estimators=number_of_trees,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_xgboost_classifier(training_features, training_labels,
                              validation_features, validation_labels):
    # Multi-class softmax objective with log-loss as the evaluation metric
    model = XGBClassifier(
        objective='multi:softmax',
        eval_metric='mlogloss',
        verbosity=0
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_multilayer_perceptron(training_features, training_labels,
                                 validation_features, validation_labels,
                                 hidden_layers=(256, 128), activation=nn.ReLU,
                                 dropout=0.0, epochs=200, learning_rate=1e-3):
    # GPU preferred but CPU works — feature vectors fit comfortably in memory.
    compute_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
    number_of_classes = len(np.unique(training_labels))

    training_input     = torch.tensor(np.asarray(training_features,   np.float32)).to(compute_device)
    training_targets   = torch.tensor(np.asarray(training_labels,     np.int64)).to(compute_device)
    validation_input   = torch.tensor(np.asarray(validation_features, np.float32)).to(compute_device)
    validation_targets = torch.tensor(np.asarray(validation_labels,   np.int64)).to(compute_device)

    training_data_loader = DataLoader(
        TensorDataset(training_input, training_targets),
        batch_size=16,
        shuffle=True
    )

    model         = MLP(training_input.shape[1], number_of_classes, hidden_layers, activation, dropout).to(compute_device)
    optimizer     = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()

    # Early stopping: track the best validation checkpoint and revert to it at the end
    best_validation_loss       = float('inf')
    best_model_weights         = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()

        for input_batch, target_batch in training_data_loader:
            optimizer.zero_grad()
            batch_loss = loss_function(model(input_batch), target_batch)
            batch_loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            current_validation_loss = loss_function(model(validation_input), validation_targets).item()

        if current_validation_loss < best_validation_loss:
            best_validation_loss       = current_validation_loss
            best_model_weights         = model.state_dict()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= 25:
                break

    model.load_state_dict(best_model_weights)
    model.eval()

    with torch.no_grad():
        predictions = model(validation_input).argmax(dim=1).cpu().numpy()

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_knn_classifier(training_features, training_labels,
                          validation_features, validation_labels,
                          number_of_neighbours=10, distance_metric='cosine'):
    # Cosine distance outperforms Euclidean for high-dimensional backbone embeddings.
    # DINOv2 and ConvNeXt features lie on roughly spherical manifolds, so
    # angular separation is more discriminative than absolute magnitude differences.
    model = KNeighborsClassifier(
        n_neighbors=number_of_neighbours,
        metric=distance_metric,
        n_jobs=-1
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_lightgbm_classifier(training_features, training_labels,
                               validation_features, validation_labels,
                               number_of_estimators=500, max_leaf_nodes=63,
                               learning_rate=0.05):
    # Histogram-based gradient boosting — typically faster than XGBoost
    # with comparable accuracy on tabular feature data.
    model = LGBMClassifier(
        n_estimators=number_of_estimators,
        num_leaves=max_leaf_nodes,
        learning_rate=learning_rate,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_catboost_classifier(training_features, training_labels,
                               validation_features, validation_labels,
                               number_of_iterations=500, tree_depth=6,
                               learning_rate=0.05):
    # Ordered boosting with strong defaults — less hand-tuning needed than XGBoost.
    # CatBoost predict() returns float, so we cast to int for metric compatibility.
    model = CatBoostClassifier(
        iterations=number_of_iterations,
        depth=tree_depth,
        learning_rate=learning_rate,
        verbose=0,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64)).astype(int)

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )


def train_stacking_ensemble(training_features, training_labels,
                             validation_features, validation_labels):
    # Stacking: base learners generate out-of-fold meta-features via 5-fold CV,
    # then a LogisticRegression meta-learner classifies on those meta-features.
    # Typically +0.5–1% F1 over the best individual model.
    #
    # Base learners use n_jobs=1 to prevent nested parallelism deadlocks when
    # StackingClassifier already parallelises across CV folds with n_jobs=-1.
    base_estimators = [
        ('svm',      SVC(kernel='rbf', probability=True, decision_function_shape='ovr')),
        ('lightgbm', LGBMClassifier(n_estimators=300, num_leaves=63, verbose=-1, n_jobs=1)),
        ('knn',      KNeighborsClassifier(n_neighbors=10, metric='cosine', n_jobs=1)),
    ]

    model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(max_iter=2000),
        cv=5,
        n_jobs=-1,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

In [ ]:
def finetune(images_file, train_idx, val_idx, num_classes,
             num_blocks=None, backbone_lr=1e-5, head_lr=1e-3,
             dropout=0.1, epochs=20, batch_size=32,
             patience=5, cache_frozen=True):
    # Fine-tune the last few backbone blocks on your training data.
    # backbone_model must already be loaded (from DEFINITIONS).
    # GPU strongly recommended.
    #
    # Speed / RAM optimisations:
    #   cache_frozen=True  Pre-computes frozen-layer activations once; training only
    #                      runs the small unfrozen suffix each step (~5-10x faster).
    #                      Images are mmap'd (not fully loaded) and freed after caching,
    #                      so peak RAM stays low. Cache stored in fp16 (~5.4 GB for
    #                      convnextv2, ~3 GB for dinov2). Set False if still OOM.
    #   mixed precision    Forward/backward run in fp16 on the GPU (~2x faster).
    #
    # num_blocks=None picks a sensible default per model:
    #   dinov2     : 4 of 40 blocks (~10%)
    #   convnextv2 : 3 of  3 last-stage blocks (the entire final stage)

    print(f'Device: {device}  |  backbone: {FEATURE_EXTRACTOR}  (GPU strongly recommended)')

    # mmap=True: tensors are memory-mapped from disk, not fully loaded into RAM upfront.
    # Pages are read on demand as we iterate through images in the caching loop below,
    # then can be evicted. After caching, we delete the reference to free the pages.
    data   = torch.load(images_file, weights_only=False, mmap=True)
    images = data['tensors']
    labels = data['labels']

    # --- Step 1: Freeze the whole backbone -----------------------------------
    for param in backbone_model.parameters():
        param.requires_grad = False

    # --- Step 2: Identify blocks and norm for this backbone ------------------
    if FEATURE_EXTRACTOR == 'dinov2':
        blocks         = list(backbone_model.blocks)             # 40 transformer blocks total
        norm           = backbone_model.norm
        default_blocks = 4                                       # ~10% of 40
    elif FEATURE_EXTRACTOR == 'convnextv2':
        blocks         = list(backbone_model.stages[-1].blocks)  # last stage: 3 conv blocks
        norm           = backbone_model.norm_pre
        default_blocks = len(blocks)                             # all 3
    if num_blocks is None:
        num_blocks = default_blocks

    # --- Step 3: Unfreeze last num_blocks + norm ------------------------------
    for block in blocks[-num_blocks:]:
        for param in block.parameters():
            param.requires_grad = True
    for param in norm.parameters():
        param.requires_grad = True
    trainable_parameter_count = sum(p.numel() for p in backbone_model.parameters() if p.requires_grad)
    total_parameter_count     = sum(p.numel() for p in backbone_model.parameters())
    print(f'Trainable: {trainable_parameter_count:,}/{total_parameter_count:,} '
          f'({100*trainable_parameter_count/total_parameter_count:.1f}%)  '
          f'[{num_blocks}/{len(blocks)} blocks unfrozen]')

    # --- Step 4: Define frozen prefix and trainable suffix -------------------
    # frozen_prefix : the layers we are NOT updating — their output never changes.
    # suffix        : the unfrozen layers — all we need to run per training step.

    def frozen_prefix(batch):
        # Run the backbone up to (but not including) the unfrozen blocks.
        if FEATURE_EXTRACTOR == 'dinov2':
            x = backbone_model.prepare_tokens_with_masks(batch)  # patch embed + pos embed
            for block in backbone_model.blocks[:-num_blocks]:
                x = block(x)
        elif FEATURE_EXTRACTOR == 'convnextv2':
            x = backbone_model.stem(batch)
            for stage in backbone_model.stages[:-1]:              # stages 0-2 (frozen)
                x = stage(x)
            x = backbone_model.stages[-1].downsample(x)           # frozen spatial downsample
        return x

    def suffix(x):
        # Run only the unfrozen layers; return the feature vector.
        if FEATURE_EXTRACTOR == 'dinov2':
            for block in blocks[-num_blocks:]:
                x = block(x)
            x = backbone_model.norm(x)
            return x[:, 0]            # CLS token → (B, 1536)
        elif FEATURE_EXTRACTOR == 'convnextv2':
            for block in blocks[-num_blocks:]:
                x = block(x)
            x = backbone_model.norm_pre(x)
            return x.mean([-2, -1])   # global average pool → (B, 2816)

    # --- Step 5: Pre-compute frozen activations (one-time cost) --------------
    # Run all images through frozen_prefix once and cache in fp16.
    # fp16 halves RAM vs fp32 (~5.4 GB instead of ~10.7 GB for convnextv2).
    # After caching, delete image tensors — training only needs the cache.
    if cache_frozen:
        with torch.no_grad():
            sample_activation = frozen_prefix(images[:1].to(device))
        estimated_cache_ram_gb = sample_activation.numel() * len(images) * 2 / 1e9  # 2 bytes = fp16
        print(f'Caching frozen activations (~{estimated_cache_ram_gb:.1f} GB RAM in fp16)...')
        cached_activation_chunks = []
        backbone_model.eval()
        with torch.no_grad():
            for i in range(0, len(images), batch_size):
                # mmap: only this chunk's pages are in RAM right now
                cached_activation_chunks.append(
                    frozen_prefix(images[i:i+batch_size].to(device)).cpu().half()
                )
                print(f'  {min(i+batch_size, len(images))}/{len(images)}', end='\r')
        frozen_cache = torch.cat(cached_activation_chunks)
        # Free image tensors — mmap pages can now be evicted, reclaiming RAM
        del cached_activation_chunks, data, images
        print(f'\n  Done. Cache shape: {tuple(frozen_cache.shape)}  (fp16)')
        train_dataset = TensorDataset(frozen_cache[train_idx], labels[train_idx])
        val_source    = frozen_cache[val_idx]
    else:
        # No caching — run full backbone each step (slower, uses more RAM)
        train_dataset = TensorDataset(images[train_idx], labels[train_idx])
        val_source    = images[val_idx]

    training_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # --- Step 6: Classification head -----------------------------------------
    with torch.no_grad():
        # .float() because this runs outside autocast — model expects fp32 input
        sample_input      = (frozen_cache if cache_frozen else images)[:1].to(device).float()
        feature_dimension = (suffix(sample_input) if cache_frozen else backbone_model(sample_input)).shape[-1]
    head = nn.Sequential(
        nn.Linear(feature_dimension, 512), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    ).to(device)

    # Backbone params use a much smaller lr to preserve pre-trained knowledge.
    optimizer = torch.optim.AdamW([
        {'params': [p for p in backbone_model.parameters() if p.requires_grad], 'lr': backbone_lr},
        {'params': head.parameters(), 'lr': head_lr},
    ], weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()

    # Mixed precision: matmuls run in fp16 — ~2x faster, no meaningful accuracy loss.
    is_mixed_precision_enabled = (device == 'cuda')
    gradient_scaler            = torch.amp.GradScaler('cuda', enabled=is_mixed_precision_enabled)
    # Load cache slices as fp16 on GPU (matches autocast); fp32 on CPU fallback.
    batch_load_dtype = torch.float16 if is_mixed_precision_enabled else torch.float32

    trainable_param_names = {name for name, p in backbone_model.named_parameters() if p.requires_grad}

    # Helper: run source (cached acts or raw images) through the right forward fn.
    # Always uses fp32 since it runs outside the autocast training context.
    def extract_features_from_source(source):
        feature_chunks = []
        with torch.no_grad():
            for i in range(0, len(source), batch_size):
                chunk = source[i:i+batch_size].to(device=device, dtype=torch.float32)
                feature_chunks.append(
                    (suffix(chunk) if cache_frozen else backbone_model(chunk)).cpu()
                )
        return torch.cat(feature_chunks)

    # --- Step 7: Training loop with early stopping ---------------------------
    best_validation_loss           = float('inf')
    best_backbone_state            = None
    best_classification_head_state = None
    epochs_without_improvement     = 0

    for epoch in range(epochs):
        backbone_model.train()
        head.train()
        for input_batch, target_batch in training_data_loader:
            # fp16 on GPU (inside autocast), fp32 on CPU
            input_batch  = input_batch.to(device=device, dtype=batch_load_dtype)
            target_batch = target_batch.to(device)
            optimizer.zero_grad()
            with torch.autocast('cuda', enabled=is_mixed_precision_enabled):  # fp16 forward + backward
                batch_features = suffix(input_batch) if cache_frozen else backbone_model(input_batch)
                batch_loss     = loss_function(head(batch_features), target_batch)
            gradient_scaler.scale(batch_loss).backward()
            gradient_scaler.step(optimizer)
            gradient_scaler.update()

        backbone_model.eval()
        head.eval()
        validation_features     = extract_features_from_source(val_source).to(device)
        current_validation_loss = loss_function(
            head(validation_features), labels[val_idx].to(device)
        ).item()

        print(f'Epoch {epoch+1:>3}/{epochs}  val_loss={current_validation_loss:.4f}', end='\r')

        if current_validation_loss < best_validation_loss:
            best_validation_loss = current_validation_loss
            # Only save the unfrozen params — avoids duplicating the whole model
            best_backbone_state = {
                k: v.cpu().clone()
                for k, v in backbone_model.state_dict().items()
                if k in trainable_param_names
            }
            best_classification_head_state = {
                k: v.cpu().clone() for k, v in head.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break

    print(f'\nBest val_loss: {best_validation_loss:.4f}')

    # --- Step 8: Restore best weights ----------------------------------------
    current_backbone_state = backbone_model.state_dict()
    current_backbone_state.update({k: v.to(device) for k, v in best_backbone_state.items()})
    backbone_model.load_state_dict(current_backbone_state)
    head.load_state_dict({k: v.to(device) for k, v in best_classification_head_state.items()})

    # Final val predictions with restored weights
    backbone_model.eval()
    head.eval()
    validation_predictions = (
        head(extract_features_from_source(val_source).to(device)).argmax(dim=1).cpu().numpy()
    )

    # Re-freeze backbone so subsequent extract() calls behave correctly
    for param in backbone_model.parameters():
        param.requires_grad = False

    return head, compute_classification_metrics(labels[val_idx].numpy(), validation_predictions)

## PREPROCESSING
Load images from disk, apply the whatever model's transform, and save as tensors.

In [ ]:
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'

if not os.path.exists(train_image_dir):
    raise FileNotFoundError(f'Training folder not found: {train_image_dir}')
if not os.path.exists(test_image_dir):
    raise FileNotFoundError(f'Test folder not found: {test_image_dir}')

# Save preprocessed tensors to disk (only needs to run once per backbone)
# The feat_suffix keeps dinov2 and convnextv2 files separate.
save_images(train_image_dir, f'Data\\task1_data\\t1_train{feat_suffix}.pt', has_labels=True)
save_images(test_image_dir,  f'Data\\task1_data\\t1_test{feat_suffix}.pt',  has_labels=False)

## FEATURE EXTRACTION
Load raw images and run them through **both** backbones in one pass.
Each model applies its own preprocessing so features are always correct.
Only needs to run once; re-run only if the image files change.

In [ ]:
# Extract features for both DINOv2 and ConvNeXt V2 in one pass.
# Each model uses its own preprocessing so features are always correct.
# Saves separate feature files; LOAD DATA picks the right one via feat_suffix.

import timm

compute_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# DINOv2 preprocessing (standard ImageNet stats, 224px center crop)
dinov2_image_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ConvNeXt V2 preprocessing — timm resolves the exact crop size and norm for this checkpoint
convnextv2_temp_model      = timm.create_model('convnextv2_huge.fcmae_ft_in22k_in1k_512', pretrained=False, num_classes=0)
convnextv2_data_config     = timm.data.resolve_model_data_config(convnextv2_temp_model)
convnextv2_image_transform = timm.data.create_transform(**convnextv2_data_config, is_training=False)
del convnextv2_temp_model  # only needed to read the config

# Each entry: (backbone_name, file_suffix, image_transform, load_backbone_fn, encode_fn)
BACKBONE_CONFIGS = [
    ('dinov2',     '',            dinov2_image_transform,
     lambda: torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14'),
     lambda model, batch: model(batch).float()),
    ('convnextv2', '_convnextv2', convnextv2_image_transform,
     lambda: timm.create_model('convnextv2_huge.fcmae_ft_in22k_in1k_512', pretrained=True, num_classes=0),
     lambda model, batch: model(batch).float()),
]


def load_images_from_dir(image_dir, image_transform, has_labels):
    # Load every image from image_dir, apply image_transform.
    # Returns (image_tensors, label_tensor_or_None, file_paths, class_to_index_dict).
    image_tensors_list = []
    label_list         = []
    file_paths         = []
    class_to_index     = {}

    if has_labels:
        class_names    = sorted(os.listdir(image_dir))
        class_to_index = {cls: i for i, cls in enumerate(class_names)}
        total_images   = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )
        image_count = 0
        for class_name in class_names:
            for filename in os.listdir(os.path.join(image_dir, class_name)):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                image_path = os.path.join(image_dir, class_name, filename)
                image_tensors_list.append(image_transform(Image.open(image_path).convert('RGB')))
                label_list.append(class_to_index[class_name])
                file_paths.append(image_path)
                image_count += 1
                print(f'  {image_count}/{total_images}', end='\r')
    else:
        image_files = sorted(
            fn for fn in os.listdir(image_dir)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png'))
        )
        for i, filename in enumerate(image_files, 1):
            image_path = os.path.join(image_dir, filename)
            image_tensors_list.append(image_transform(Image.open(image_path).convert('RGB')))
            file_paths.append(image_path)
            print(f'  {i}/{len(image_files)}', end='\r')

    label_tensor = torch.tensor(label_list) if label_list else None
    return torch.stack(image_tensors_list), label_tensor, file_paths, class_to_index


def extract_and_save_features(backbone_model_instance, encode_function, image_tensors,
                               label_tensor, file_paths, class_to_index, output_file):
    # Run image_tensors through backbone_model_instance and save features + metadata.
    feature_vectors = []
    with torch.no_grad():
        for i, tensor in enumerate(image_tensors, 1):
            feature_vectors.append(
                encode_function(backbone_model_instance, tensor.unsqueeze(0).to(compute_device))
                .squeeze().cpu().numpy()
            )
            print(f'  {i}/{len(image_tensors)}', end='\r')

    save_dict = {
        'tensors':  image_tensors,
        'paths':    file_paths,
        'features': np.array(feature_vectors),
    }
    if label_tensor is not None:
        save_dict['labels']         = label_tensor
        save_dict['class_to_index'] = class_to_index
    torch.save(save_dict, output_file)
    print(f'\n  Saved {np.array(feature_vectors).shape} -> {output_file}')


# Main extraction loop ---------------------------------------------------------
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'

for backbone_name, backbone_suffix, image_transform, load_backbone_fn, encode_fn in BACKBONE_CONFIGS:
    print(f'\n=== {backbone_name} (device={compute_device}) ===')
    backbone_model_instance = load_backbone_fn()
    backbone_model_instance = backbone_model_instance.to(compute_device).eval()

    print('  train images:')
    train_image_tensors, train_label_tensors, train_file_paths, train_class_to_index = \
        load_images_from_dir(train_image_dir, image_transform, has_labels=True)
    extract_and_save_features(
        backbone_model_instance, encode_fn,
        train_image_tensors, train_label_tensors, train_file_paths, train_class_to_index,
        f'Data\\task1_data\\t1_train_features{backbone_suffix}.pt'
    )

    print('  test images:')
    test_image_tensors, _, test_file_paths, _ = \
        load_images_from_dir(test_image_dir, image_transform, has_labels=False)
    extract_and_save_features(
        backbone_model_instance, encode_fn,
        test_image_tensors, None, test_file_paths, {},
        f'Data\\task1_data\\t1_test_features{backbone_suffix}.pt'
    )

    del backbone_model_instance
    if compute_device == 'cuda':
        torch.cuda.empty_cache()

print('\nAll done — feature files ready for dinov2 / convnextv2.')

## FINE-TUNING
Optionally fine-tune the last few backbone blocks on your data, then re-extract features.
Supports both backbones: `dinov2` and `convnextv2`.
Skip this section if you want to use frozen features only.

In [7]:
# Fine-tune the last few backbone blocks on task 1 training images.
# After this runs, re-extract features from the adapted backbone so the rest of
# the notebook (LOAD DATA, EVAL, HYPER SEARCH) automatically benefits.
#
# GPU strongly recommended - on CPU this will take a long time.
#
# IMPORTANT: The PREPROCESSING cell must have been run with the current
# FEATURE_EXTRACTOR selected so that t1_train{feat_suffix}.pt uses the correct preprocessing.

raw_pt      = torch.load(f'Data\\task1_data\\t1_train{feat_suffix}.pt', weights_only=False)
num_classes = len(np.unique(raw_pt['labels'].numpy()))
all_idx     = np.arange(len(raw_pt['labels']))
train_idx_ft, val_idx_ft = train_test_split(
    all_idx, test_size=0.2, random_state=42,
    stratify=raw_pt['labels'].numpy()
)

fine_tuned_head, ft_metrics = finetune(
    images_file = f'Data\\task1_data\\t1_train{feat_suffix}.pt',
    train_idx   = train_idx_ft,
    val_idx     = val_idx_ft,
    num_classes = num_classes,
    num_blocks  = None,
    backbone_lr = 1e-5,
    head_lr     = 1e-3,
    dropout     = 0.1,
    epochs      = 20,
    batch_size  = 8,    # keep small — caching pass runs the full frozen backbone
                        # which has huge intermediates at 512px (convnextv2 stage 0
                        # expands to 4x channels at 128x128). 8 is safe on ~12 GB VRAM.
    cache_frozen=True,
)

print(f'Fine-tuned val:  F1={ft_metrics["f1"]:.4f}  Acc={ft_metrics["accuracy"]:.4f}')

# Re-extract features using the now-adapted backbone and save to new files.
# In the LOAD DATA cell, set USE_FINETUNED = True to use these.
print('\nRe-extracting features with fine-tuned backbone...')
extract(f'Data\\task1_data\\t1_train{feat_suffix}.pt', f'Data\\task1_data\\t1_train_features_ft{feat_suffix}.pt')
extract(f'Data\\task1_data\\t1_test{feat_suffix}.pt',  f'Data\\task1_data\\t1_test_features_ft{feat_suffix}.pt')
print('Done. Set USE_FINETUNED = True in the LOAD DATA cell to use these features.')

Device: cuda  |  backbone: convnextv2  (GPU strongly recommended)
Trainable: 190,865,664/657,472,640 (29.0%)  [3/3 blocks unfrozen]
Caching frozen activations (~5.4 GB RAM in fp16)...
  3750/3750
  Done. Cache shape: (3750, 2816, 16, 16)  (fp16)


C:\Users\TheRe\AppData\Local\Temp\ipykernel_21736\3085805204.py:134: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)


Epoch   6/20  val_loss=0.1802
Early stopping at epoch 6

Best val_loss: 0.1137
Fine-tuned val:  F1=0.9694  Acc=0.9693

Re-extracting features with fine-tuned backbone...
3750/3750
Extracted (3750, 2816) -> saved to Data\task1_data\t1_train_features_ft_convnextv2.pt
1250/1250
Extracted (1250, 2816) -> saved to Data\task1_data\t1_test_features_ft_convnextv2.pt
Done. Set USE_FINETUNED = True in the LOAD DATA cell to use these features.


## LOAD DATA

In [ ]:
# Set USE_FINETUNED = True after running the FINE-TUNING cell to use the adapted features.
USE_FINETUNED = True

if USE_FINETUNED:
    train_data = torch.load(f'Data\\task1_data\\t1_train_features_ft{feat_suffix}.pt', weights_only=False)
    test_data  = torch.load(f'Data\\task1_data\\t1_test_features_ft{feat_suffix}.pt',  weights_only=False)
else:
    train_data = torch.load(f'Data\\task1_data\\t1_train_features{feat_suffix}.pt', weights_only=False)
    test_data  = torch.load(f'Data\\task1_data\\t1_test_features{feat_suffix}.pt',  weights_only=False)

train_features = train_data['features']        # shape: (N, D)
train_labels   = train_data['labels'].numpy()  # shape: (N,)
test_features  = test_data['features']         # shape: (M, D) — no labels

print(f'Train: {train_features.shape}  Labels: {train_labels.shape}')
print(f'Test (submission): {test_features.shape}')
print(f'Classes: {train_data["class_to_index"]}')

# Stratified 80/20 split — stratify ensures class proportions are preserved in both halves.
# This exact split (same random_state and stratify target) is reproduced in ERROR ANALYSIS.
(
    training_split_features,
    validation_split_features,
    training_split_labels,
    validation_split_labels,
) = train_test_split(
    train_features,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

print(f'\nTraining split:   {training_split_features.shape}')
print(f'Validation split: {validation_split_features.shape}')

## MODEL EVALUATION

In [ ]:
# --- Settings ----------------------------------------------------------------
SVM_KERNEL     = 'rbf'   # 'linear' | 'rbf' | 'poly'
KNN_NEIGHBOURS = 10      # number of nearest neighbours (cosine distance)
# -----------------------------------------------------------------------------

cross_validation_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Running 5-fold cross validation...\n')

# Logistic Regression
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_logistic_regression(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report('Logistic Regression (5-fold CV)', fold_metrics_list)

# SVM
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_support_vector_machine(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
        kernel=SVM_KERNEL,
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report(f'SVM kernel={SVM_KERNEL} (5-fold CV)', fold_metrics_list)

# kNN (cosine distance)
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_knn_classifier(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
        number_of_neighbours=KNN_NEIGHBOURS,
        distance_metric='cosine',
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report(f'kNN k={KNN_NEIGHBOURS} cosine (5-fold CV)', fold_metrics_list)

# XGBoost
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_xgboost_classifier(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report('XGBoost (5-fold CV)', fold_metrics_list)

# LightGBM
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_lightgbm_classifier(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report('LightGBM (5-fold CV)', fold_metrics_list)

# CatBoost
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_catboost_classifier(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report('CatBoost (5-fold CV)', fold_metrics_list)

# Stacking (SVM + LightGBM + kNN → LogReg meta-learner)
# Note: slower than the others — each base learner trains across 5 folds
fold_metrics_list = []
for fold_training_indices, fold_validation_indices in cross_validation_splitter.split(train_features, train_labels):
    _, fold_metrics = train_stacking_ensemble(
        train_features[fold_training_indices], train_labels[fold_training_indices],
        train_features[fold_validation_indices], train_labels[fold_validation_indices],
    )
    fold_metrics_list.append(fold_metrics)
print_cross_validation_report('Stacking SVM+LightGBM+kNN → LogReg (5-fold CV)', fold_metrics_list)

## HYPER PARAM SEARCH

In [ ]:
import json

search_results = []


def fit_and_compute_validation_f1(model, training_features, training_labels,
                                   validation_features, validation_labels):
    # Force raw NumPy arrays to prevent LightGBM feature-name warnings
    # that arise when the input has pandas column names attached.
    clean_training_features   = np.asarray(training_features)
    clean_validation_features = np.asarray(validation_features)
    clean_training_labels     = np.asarray(training_labels)
    clean_validation_labels   = np.asarray(validation_labels)

    model.fit(clean_training_features, clean_training_labels)
    predictions = model.predict(clean_validation_features)

    return f1_score(clean_validation_labels, predictions, average='weighted')


# --- kNN search --------------------------------------------------------------
knn_configs = [
    {'k': k, 'metric': distance_metric}
    for k in [5, 10, 15, 20]
    for distance_metric in ['cosine', 'euclidean']
]

print(f'kNN search: {len(knn_configs)} configs')

for params in tqdm.tqdm(knn_configs, desc='kNN'):
    model = KNeighborsClassifier(
        n_neighbors=params['k'],
        metric=params['metric'],
        n_jobs=-1
    )

    validation_f1_score = fit_and_compute_validation_f1(
        model,
        training_split_features, training_split_labels,
        validation_split_features, validation_split_labels,
    )

    search_results.append({
        'model':       'kNN',
        'params':      f"k={params['k']}  metric={params['metric']}",
        'params_json': json.dumps({'k': params['k'], 'metric': params['metric']}),
        'f1':          validation_f1_score,
    })


# --- LightGBM search ---------------------------------------------------------
lgbm_hyperparameter_grid = {
    'n_estimators':  [200, 500],
    'num_leaves':    [31, 63, 127],
    'learning_rate': [0.05, 0.1],
}

lgbm_configs = list(ParameterGrid(lgbm_hyperparameter_grid))

print(f'\nLightGBM search: {len(lgbm_configs)} configs')

for params in tqdm.tqdm(lgbm_configs, desc='LightGBM'):
    model = LGBMClassifier(
        **params,
        n_jobs=-1,
        verbose=-1,
    )

    validation_f1_score    = fit_and_compute_validation_f1(
        model,
        training_split_features, training_split_labels,
        validation_split_features, validation_split_labels,
    )
    hyperparameter_description = '  '.join(f'{k}={v}' for k, v in sorted(params.items()))

    search_results.append({
        'model':       'LightGBM',
        'params':      hyperparameter_description,
        'params_json': json.dumps(params),
        'f1':          validation_f1_score,
    })


# --- CatBoost search ---------------------------------------------------------
catboost_hyperparameter_grid = {
    'iterations':    [200, 500],
    'depth':         [4, 6],
    'learning_rate': [0.05, 0.1],
}

catboost_configs = list(ParameterGrid(catboost_hyperparameter_grid))

print(f'\nCatBoost search: {len(catboost_configs)} configs')

for params in tqdm.tqdm(catboost_configs, desc='CatBoost'):
    model = CatBoostClassifier(
        **params,
        verbose=0,
    )

    validation_f1_score    = fit_and_compute_validation_f1(
        model,
        training_split_features, training_split_labels,
        validation_split_features, validation_split_labels,
    )
    hyperparameter_description = '  '.join(f'{k}={v}' for k, v in sorted(params.items()))

    search_results.append({
        'model':       'CatBoost',
        'params':      hyperparameter_description,
        'params_json': json.dumps(params),
        'f1':          validation_f1_score,
    })


# --- SVM search --------------------------------------------------------------
svm_hyperparameter_grid = [
    {'kernel': ['linear'], 'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
    {'kernel': ['rbf'],    'C': [0.01, 0.1, 1, 10, 100, 1000],
                            'gamma': ['scale', 'auto', 0.1, 0.01, 0.001, 0.0001]},
    {'kernel': ['poly'],   'C': [0.01, 0.1, 1, 10, 100],
                            'gamma': ['scale', 'auto', 0.01, 0.001],
                            'degree': [2, 3, 4, 5]},
    {'kernel': ['sigmoid'], 'C': [0.01, 0.1, 1, 10, 100],
                             'gamma': ['scale', 'auto', 0.01, 0.001]},
]

svm_configs = list(ParameterGrid(svm_hyperparameter_grid))

print(f'\nSVM search: {len(svm_configs)} configs')

for params in tqdm.tqdm(svm_configs, desc='SVM'):
    model = SVC(
        decision_function_shape='ovr',
        **params,
    )

    validation_f1_score    = fit_and_compute_validation_f1(
        model,
        training_split_features, training_split_labels,
        validation_split_features, validation_split_labels,
    )
    hyperparameter_description = '  '.join(f'{k}={v}' for k, v in sorted(params.items()))

    search_results.append({
        'model':       'SVM',
        'params':      hyperparameter_description,
        'params_json': json.dumps(params),
        'f1':          validation_f1_score,
    })


# --- MLP search --------------------------------------------------------------
# Search over a full grid of hidden layer sizes (1-, 2-, and 3-layer networks)
HIDDEN_LAYER_SIZES = [1024, 512, 256]

hidden_layer_combinations = (
    [(size,) for size in HIDDEN_LAYER_SIZES]
    + [(a, b) for a in HIDDEN_LAYER_SIZES for b in HIDDEN_LAYER_SIZES]
    + [(a, b, c) for a in HIDDEN_LAYER_SIZES for b in HIDDEN_LAYER_SIZES for c in HIDDEN_LAYER_SIZES]
)

mlp_configs = list(product(
    hidden_layer_combinations,
    [nn.GELU, nn.ReLU, nn.SiLU],
    [0.0, 0.2, 0.4],
    [1e-2, 1e-4],
))

print(f'\nMLP search: {len(mlp_configs)} configs')

for hidden_layers, activation, dropout, learning_rate in tqdm.tqdm(mlp_configs, desc='MLP'):
    #continue  # remove this line to run the MLP search

    _, fold_metrics = train_multilayer_perceptron(
        training_split_features, training_split_labels,
        validation_split_features, validation_split_labels,
        hidden_layers=hidden_layers,
        activation=activation,
        dropout=dropout,
        learning_rate=learning_rate,
    )

    hyperparameter_description = (
        f'layers={hidden_layers}  '
        f'act={activation.__name__}  '
        f'drop={dropout}  '
        f'lr={learning_rate}'
    )

    search_results.append({
        'model':       'MLP',
        'params':      hyperparameter_description,
        'params_json': json.dumps({
            'layers': str(hidden_layers),
            'act':    activation.__name__,
            'drop':   dropout,
            'lr':     learning_rate,
        }),
        'f1': fold_metrics['f1'],
    })


# --- Save and display results ------------------------------------------------
results_dataframe = (
    pd.DataFrame(search_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

fine_tuning_suffix         = 'ft' if USE_FINETUNED else 'noft'
search_results_csv_filename = f'hyperparameter_search_results_{FEATURE_EXTRACTOR}_{fine_tuning_suffix}.csv'

results_dataframe.to_csv(f'Data/task1_data/{search_results_csv_filename}', index=False)
print(f'Saved full results to Data/task1_data/{search_results_csv_filename}')

loaded_results_dataframe = pd.read_csv(f'Data/task1_data/{search_results_csv_filename}')

NUMBER_OF_TOP_RESULTS_TO_DISPLAY = 100

W = 90
print(f'\n{"="*W}')
print(f'   TOP {NUMBER_OF_TOP_RESULTS_TO_DISPLAY} — {len(loaded_results_dataframe)} configs evaluated, ranked by weighted F1')
print(f'{"="*W}')
print(f'   {"#":<3}  {"Model":<10}  {"F1":<8}  Params')
print(f'   {"-"*(W-4)}')

for rank, row in loaded_results_dataframe.head(NUMBER_OF_TOP_RESULTS_TO_DISPLAY).iterrows():
    print(f'   {rank+1:<3}  {row["model"]:<10}  {row["f1"]:.4f}    {row["params"]}')

print(f'{"="*W}')
print(f'\nFull results available in Data/task1_data/{search_results_csv_filename} and results_dataframe')

## SUBMISSION

In [ ]:
# --- Settings ----------------------------------------------------------------
# Choose a model and paste in the best params from the search above.
SUBMISSION_MODEL = 'linear'   # 'linear' | 'svm' | 'knn' | 'xgboost' | 'lgbm' | 'catboost' | 'mlp' | 'stack'

SVM_SUBMISSION_PARAMS        = {'kernel': 'rbf', 'C': 1, 'gamma': 0.01}
KNN_NEIGHBOURS               = 10
KNN_DISTANCE_METRIC          = 'cosine'
LIGHTGBM_SUBMISSION_PARAMS   = {'n_estimators': 500, 'num_leaves': 63, 'learning_rate': 0.05}
CATBOOST_SUBMISSION_PARAMS   = {'iterations': 500, 'depth': 6, 'learning_rate': 0.05}

MLP_SUBMISSION_HIDDEN_LAYERS  = (512, 256)
MLP_SUBMISSION_DROPOUT_RATE   = 0.2
MLP_SUBMISSION_EPOCHS         = 200
MLP_SUBMISSION_LEARNING_RATE  = 3e-4
# -----------------------------------------------------------------------------

all_training_features    = np.asarray(train_features, np.float64)
all_training_labels      = np.asarray(train_labels,   np.int64)
test_submission_features = np.asarray(test_features,  np.float64)

print(f'Training {SUBMISSION_MODEL} on all {len(all_training_features)} labelled samples...')

if SUBMISSION_MODEL == 'linear':
    model = LogisticRegression(max_iter=2000)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'svm':
    model = SVC(decision_function_shape='ovr', **SVM_SUBMISSION_PARAMS)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'knn':
    model = KNeighborsClassifier(n_neighbors=KNN_NEIGHBOURS, metric=KNN_DISTANCE_METRIC, n_jobs=-1)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'xgboost':
    model = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'lgbm':
    model = LGBMClassifier(**LIGHTGBM_SUBMISSION_PARAMS, n_jobs=-1, verbose=-1)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'catboost':
    model = CatBoostClassifier(**CATBOOST_SUBMISSION_PARAMS, verbose=0)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features).astype(int)

elif SUBMISSION_MODEL == 'stack':
    # Hold out 5% for the stacker's internal CV to prevent the meta-learner from over-fitting
    holdout_training_features, holdout_validation_features, holdout_training_labels, holdout_validation_labels = train_test_split(
        train_features, train_labels, test_size=0.05, random_state=42, stratify=train_labels
    )
    model, _ = train_stacking_ensemble(
        holdout_training_features, holdout_training_labels,
        holdout_validation_features, holdout_validation_labels,
    )
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'mlp':
    # Hold out 5% for early stopping — MLP needs a validation signal during training
    holdout_training_features, holdout_validation_features, holdout_training_labels, holdout_validation_labels = train_test_split(
        train_features, train_labels, test_size=0.05, random_state=42, stratify=train_labels
    )
    model, _ = train_multilayer_perceptron(
        holdout_training_features, holdout_training_labels,
        holdout_validation_features, holdout_validation_labels,
        hidden_layers=MLP_SUBMISSION_HIDDEN_LAYERS,
        dropout=MLP_SUBMISSION_DROPOUT_RATE,
        epochs=MLP_SUBMISSION_EPOCHS,
        learning_rate=MLP_SUBMISSION_LEARNING_RATE,
    )
    model.eval()
    with torch.no_grad():
        mlp_inference_device  = next(model.parameters()).device
        test_features_tensor  = torch.tensor(np.asarray(test_features, np.float32)).to(mlp_inference_device)
        predictions           = model(test_features_tensor).argmax(dim=1).cpu().numpy()

image_ids  = [os.path.splitext(os.path.basename(path))[0] for path in test_data['paths']]
submission = pd.DataFrame({'image_id': image_ids, 'class_id': predictions})
submission.to_csv('t1_submission.csv', index=False)
print(f'Saved {len(submission)} predictions to t1_submission.csv')